# LLM 5일차 (1) — ChromaDB 메타데이터 필터 연산자 · FAISS vs ChromaDB · CSV 불러오기

교재 `AI Agent` **p.51~54**(ChromaDB 메타데이터 필터링 예시 · 연산자 정리 · MySQL 문법 비교 · 실습 결론)를 수업 데이터 `company_policy.json` 으로 실행하고, 수업 필기 `RAG_실습6`(`CSVLoader` 로 챗봇 데이터 불러오기)을 정리 · 개선한 노트입니다.

```python
retriever = vectorstore.as_retriever(search_kwargs={'k': 1, 'filter': {
    '$and': [{'category': {'$eq': '보안'}}, {'year': {'$eq': 2026}}]}})      # WHERE category='보안' AND year=2026 LIMIT 1
```

- BM25 희소 검색은 [`bm25_sparse_search.ipynb`](./bm25_sparse_search.ipynb), 하이브리드 검색 · RRF 는 [`hybrid_search_rrf.ipynb`](./hybrid_search_rrf.ipynb)
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다(Day 046 ~ 049 와 같은 방식).
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | OpenAI 대신 **로컬 임베딩 모델**(`jhgan/ko-sroberta-multitask`) 또는 **가짜 LLM** 으로 실제로 돌린 셀 — 밀집 검색 순위 · 점수는 수업의 `text-embedding-3-small` 과 다를 수 있다 |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 (필터 · BM25 · 형태소 분석 · RRF 손계산 · 수업 기록 분석) — **BM25 는 임베딩이 필요 없어서 수업과 같은 결과** |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langchain-chroma` · `chromadb` | `Chroma` · 메타데이터 필터 | `pip install langchain-chroma` |
| `langchain-community` · `faiss-cpu` | `FAISS` · `CSVLoader` | `pip install langchain-community faiss-cpu` |
| `langchain-huggingface` · `sentence-transformers` | 🧪 로컬 한국어 임베딩 | `pip install langchain-huggingface sentence-transformers` |
| `langchain-openai` · `python-dotenv` · `pandas` | ⏳ `OpenAIEmbeddings` · 키 · 표 | `pip install langchain-openai python-dotenv pandas` |

**데이터** : 노트북과 같은 폴더의 `company_policy.json`(사내 규정 5개 + 메타데이터) · `ChatBotData_sample.csv`(챗봇 Q · A 30줄). **실행** : 맨 위부터 모두 실행 — 🧪 로컬 임베딩은 처음 한 번 모델(약 440MB)을 받는다. 연습용 DB 폴더(`chroma_db_*`)는 첫 셀이 지우고 새로 만든다.

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langchain-chroma langchain-community faiss-cpu langchain-huggingface sentence-transformers langchain-openai python-dotenv pandas

In [2]:
import os, json, time, shutil, statistics, warnings
import pandas as pd
warnings.filterwarnings('ignore', message='.*sunset.*')
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import CSVLoader
from langchain_openai import OpenAIEmbeddings

for d in ['chroma_db_policy', 'chroma_db_csv_local', 'chroma_db_speed']:
    shutil.rmtree(d, ignore_errors=True)

In [3]:
# .env 파일 → 환경 변수 (키 값은 절대 print 하지 않는다)
from dotenv import load_dotenv, find_dotenv
loaded = load_dotenv(find_dotenv(usecwd=True))
print('.env 읽음 :', loaded, '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))

.env 읽음 : False | OPENAI_API_KEY 있음 : False


## 1. 메타데이터가 붙은 문서 — `company_policy.json`

In [4]:
with open('company_policy.json', encoding='utf-8') as f:
    raw = json.load(f)                                       # [{'page_content': …, 'metadata': {…}}, …]
policies = [Document(page_content=r['page_content'], metadata=r['metadata']) for r in raw]
pd.DataFrame([{**d.metadata, '내용': d.page_content[:28] + '…'} for d in policies])

,category,department,year,내용
0,보안,IT보안팀,2026,사내 보안 규정 제5조: 업무용 PC의 비밀번호는 …
1,보안,IT보안팀,2025,사내 보안 규정 제12조: 외부 USB 메모리 및 …
2,복지,인사팀,2026,복리후생 규정 제3조: 연간 120만 원 상당의 복…
3,인사,인사팀,2026,휴가 관리 규정 제8조: 연차 휴가는 최소 3일 전…
4,보안,IT보안팀,2026,원격 근무 규정 제2조: 재택근무 신청자는 보안 V…


- JSON 의 `page_content` · `metadata` 를 그대로 `Document` 로 — 분할 없이 **규정 1개 = 문서 1개**
- `year` 는 숫자(`2026`), `category` 는 글자(`'보안'`) — 필터에 쓸 때 **자료형까지** 맞아야 한다(§3)

## 2. 연산자 — MySQL `WHERE` 와 나란히 (교재 p.51~53)

In [5]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

from langchain_huggingface import HuggingFaceEmbeddings
local_emb = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask',      # 처음 한 번 약 440MB 내려받음
                                  encode_kwargs={'normalize_embeddings': True})  # 길이 1 → 내적 = 코사인
print('벡터 길이 :', len(local_emb.embed_query('테스트')))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

벡터 길이 : 768


In [6]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

store = Chroma.from_documents(policies, local_emb, persist_directory='chroma_db_policy',
                              collection_configuration={'hnsw': {'space': 'cosine'}})
short = lambda docs: [d.page_content.split(':')[0] for d in docs]

# 교재 p.51 예시 그대로
retriever_chroma = store.as_retriever(search_kwargs={'k': 1, 'filter': {
    '$and': [{'category': {'$eq': '보안'}}, {'year': {'$eq': 2026}}]}})
print('비밀번호 규정 알려줘 →', short(retriever_chroma.invoke('비밀번호 규정 알려줘')))

비밀번호 규정 알려줘 → ['사내 보안 규정 제5조']


In [7]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

cases = [
    ("category = '보안'",                      {'category': {'$eq': '보안'}}),
    ("category != '보안'",                     {'category': {'$ne': '보안'}}),
    ('year > 2025',                            {'year': {'$gt': 2025}}),
    ('year < 2026',                            {'year': {'$lt': 2026}}),
    ("category IN ('복지', '인사')",           {'category': {'$in': ['복지', '인사']}}),
    ("category = '보안' AND year = 2026",      {'$and': [{'category': {'$eq': '보안'}}, {'year': {'$eq': 2026}}]}),
    ("category = '복지' OR year = 2025",       {'$or': [{'category': '복지'}, {'year': 2025}]}),
]
rows = []
for sql, flt in cases:
    found = store.similarity_search('규정', k=5, filter=flt)
    rows.append({'MySQL WHERE': sql, 'Chroma filter': json.dumps(flt, ensure_ascii=False), '통과한 규정': short(found)})
pd.set_option('display.max_colwidth', 80)
pd.DataFrame(rows)

,MySQL WHERE,Chroma filter,통과한 규정
0,category = '보안',"{""category"": {""$eq"": ""보안""}}","[사내 보안 규정 제5조, 사내 보안 규정 제12조, 원격 근무 규정 제2조]"
1,category != '보안',"{""category"": {""$ne"": ""보안""}}","[휴가 관리 규정 제8조, 복리후생 규정 제3조]"
2,year > 2025,"{""year"": {""$gt"": 2025}}","[사내 보안 규정 제5조, 원격 근무 규정 제2조, 휴가 관리 규정 제8조, 복리후생 규정 제3조]"
3,year < 2026,"{""year"": {""$lt"": 2026}}",[사내 보안 규정 제12조]
4,"category IN ('복지', '인사')","{""category"": {""$in"": [""복지"", ""인사""]}}","[휴가 관리 규정 제8조, 복리후생 규정 제3조]"
5,category = '보안' AND year = 2026,"{""$and"": [{""category"": {""$eq"": ""보안""}}, {""year"": {""$eq"": 2026}}]}","[사내 보안 규정 제5조, 원격 근무 규정 제2조]"
6,category = '복지' OR year = 2025,"{""$or"": [{""category"": ""복지""}, {""year"": 2025}]}","[사내 보안 규정 제12조, 복리후생 규정 제3조]"


| 할 일 | MySQL | Chroma `filter` |
|---|---|---|
| 같다 · 다르다 | `=` · `!=` | `$eq` · `$ne` (`{'category': '보안'}` 은 `$eq` 의 줄임) |
| 크다 · 크거나 같다 · 작다 · 작거나 같다 | `>` · `>=` · `<` · `<=` | `$gt` · `$gte` · `$lt` · `$lte` |
| 목록 안 · 밖 | `IN (…)` · `NOT IN (…)` | `$in` · `$nin` (교재 p.52 표에는 없음) |
| 그리고 · 또는 | `AND` · `OR` | `{'$and': [조건, 조건]}` · `{'$or': [조건, 조건]}` |
| 개수 | `LIMIT 1` | `'k': 1` — 단 SQL 의 `LIMIT` 은 순서를 정하지 않지만 `k` 는 **유사도가 높은 순서**로 1개 |

## 3. 📝 필터에서 자주 틀리는 것

In [8]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

try:                                                            # ① 조건 두 개를 $and 없이
    store.similarity_search('비밀번호', k=1, filter={'category': '보안', 'year': 2026})
except ValueError as e:
    print('① 키 두 개 :', str(e)[:70])

print("② year 를 글자 '2026' 으로 :", short(store.similarity_search('비밀번호', k=2, filter={'year': '2026'})), '← 숫자 2026 과 다르다')

print('③ 조건에 맞는 문서가 없으면 :', short(store.similarity_search('비밀번호', k=2, filter={'department': '재무팀'})), '← 빈 리스트')

① 키 두 개 : Expected where to have exactly one operator, got {'category': '보안', 'y
② year 를 글자 '2026' 으로 : [] ← 숫자 2026 과 다르다
③ 조건에 맞는 문서가 없으면 : [] ← 빈 리스트


- 필터에 걸리는 문서가 없으면 빈 리스트 — RAG 체인에서는 컨텍스트가 비게 되니 '찾을 수 없습니다' 로 답하게 해야 한다
- 메타데이터의 자료형은 **넣을 때** 정해진다. JSON 의 `2026` 은 숫자, CSV 에서 읽은 값은 모두 **글자**(§6)

## 4. FAISS 로 같은 필터 — 되기는 하지만 (교재 p.46 · 49 · 54)

In [9]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

faiss_store = FAISS.from_documents(policies, local_emb)
flt = {'$and': [{'category': {'$eq': '보안'}}, {'year': {'$eq': 2026}}]}
print('FAISS $and        :', short(faiss_store.similarity_search('비밀번호 규정 알려줘', k=1, filter=flt)))
try:
    faiss_store.similarity_search('규정', k=5, filter={'category': {'$ne': '보안'}})
except ValueError as e:
    print('FAISS $ne         :', str(e)[:60], "→ FAISS 는 '$neq'")
print('FAISS $neq        :', short(faiss_store.similarity_search('규정', k=5, filter={'category': {'$neq': '보안'}})))
print("fetch_k=1 · 복지  :", short(faiss_store.similarity_search('비밀번호', k=1, filter={'category': '복지'}, fetch_k=1)), '← 먼저 1개만 찾고 거르면 아무것도 안 남는다')
print("fetch_k=20(기본)  :", short(faiss_store.similarity_search('비밀번호', k=1, filter={'category': '복지'})))

FAISS $and        : ['사내 보안 규정 제5조']
FAISS $ne         : filter contains unsupported operator: $ne → FAISS 는 '$neq'
FAISS $neq        : ['휴가 관리 규정 제8조', '복리후생 규정 제3조']
fetch_k=1 · 복지  : [] ← 먼저 1개만 찾고 거르면 아무것도 안 남는다
fetch_k=20(기본)  : ['복리후생 규정 제3조']


- LangChain 의 FAISS 도 `$and` · `$or` · `$gt` · `$in` … 을 받는다. 다만 '다르다' 는 Chroma 와 달리 **`$neq`**
- 결정적 차이는 **언제 거르느냐** — FAISS 는 유사도로 `fetch_k`(기본 20)개를 먼저 뽑고 **그 안에서** 거른다. 조건에 맞는 문서가 그 안에 없으면 결과가 비거나 k개보다 적다. Chroma 는 조건으로 먼저 줄이고 그 안에서 찾는다

## 5. 문서가 많아지면 — 5,000개로 재 보기 (교재 p.54 실습 결론)

In [10]:
# 의미 없는 '가짜 임베딩'(같은 글 → 같은 벡터)으로 문서 5,000개 — 속도와 필터 동작만 본다
from langchain_core.embeddings import DeterministicFakeEmbedding
fake_emb = DeterministicFakeEmbedding(size=384)
many = [Document(f'규정 문서 {i}', metadata={'category': ['보안', '복지', '인사', '기술'][i % 4], 'year': 2020 + i % 7})
        for i in range(5000)]
t = time.perf_counter(); big_faiss = FAISS.from_documents(many, fake_emb); t_faiss = time.perf_counter() - t
t = time.perf_counter(); big_chroma = Chroma.from_documents(many, fake_emb, persist_directory='chroma_db_speed'); t_chroma = time.perf_counter() - t

def median_ms(fn, n=30):
    times = []
    for _ in range(n):
        start = time.perf_counter(); fn(); times.append(time.perf_counter() - start)
    return statistics.median(times) * 1000

flt = {'$and': [{'category': {'$eq': '보안'}}, {'year': {'$eq': 2026}}]}
match = sum(d.metadata['category'] == '보안' and d.metadata['year'] == 2026 for d in many)
print(f'저장 : FAISS {t_faiss:.2f}초 · Chroma {t_chroma:.2f}초')
print(f"검색 : FAISS {median_ms(lambda: big_faiss.similarity_search('규정 문서 7', k=4)):.2f}ms · Chroma {median_ms(lambda: big_chroma.similarity_search('규정 문서 7', k=4)):.2f}ms")
print(f'조건에 맞는 문서 {match}개 중 k=4 요청 → FAISS {len(big_faiss.similarity_search("규정 문서 7", k=4, filter=flt))}개 · '
      f'Chroma {len(big_chroma.similarity_search("규정 문서 7", k=4, filter=flt))}개')
print('FAISS fetch_k=5000 이면 :', len(big_faiss.similarity_search('규정 문서 7', k=4, filter=flt, fetch_k=5000)), '개')

저장 : FAISS 0.45초 · Chroma 2.94초
검색 : FAISS 0.44ms · Chroma 1.57ms
조건에 맞는 문서 178개 중 k=4 요청 → FAISS 1개 · Chroma 4개
FAISS fetch_k=5000 이면 : 4 개


- 이 환경에서 한 번 잰 값이다(실행마다 조금씩 다름). 속도는 FAISS 가 빠르지만 둘 다 ms 단위 — 이 정도 크기에서는 차이가 문제가 되지 않는다
- 문제는 **필터 결과** : 조건에 맞는 문서가 178개나 있는데 k=4 를 요청하면 FAISS 는 **1개**만 돌려준다 — 유사도로 먼저 뽑은 20개 안에 조건에 맞는 것이 1개뿐이었기 때문. `fetch_k` 를 전체 문서 수만큼 키우면 4개를 찾지만 그만큼 느려진다 — 교재 p.54 'ChromaDB 가 구조적으로 강력' 의 구체적인 뜻

## 6. CSV 불러오기 — `CSVLoader` (필기 `RAG_실습6`)

In [11]:
loader = CSVLoader(file_path='ChatBotData_sample.csv', encoding='utf-8',
                   content_columns=['Q', 'A'],          # 본문에 넣을 열
                   metadata_columns=['label'])          # 메타데이터로 뺄 열
csv_docs = loader.load()                                # 한 줄 = Document 하나
print('Document', len(csv_docs), '개')
print('본문 :', repr(csv_docs[0].page_content))
print('메타데이터 :', csv_docs[0].metadata, "← label 이 글자 '0'")
print('label 종류 :', sorted({d.metadata['label'] for d in csv_docs}), '(샘플 30줄은 모두 0 = 일상)')

Document 30 개
본문 : 'Q: 12시 땡!\nA: 하루가 또 가네요.'
메타데이터 : {'source': 'ChatBotData_sample.csv', 'row': 0, 'label': '0'} ← label 이 글자 '0'
label 종류 : ['0'] (샘플 30줄은 모두 0 = 일상)


In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

user_query="가족 여행 갈거야. 기대가 많이 되"
output = retriever.invoke(user_query)
print(f'검색 결과: \n{output}')

검색 결과: 
[Document(id='d84e328f-5539-412f-bc12-3b584137c8f0', metadata={'label': '0', 'row': 23, 'source': 'ChatBotData_sample.csv'}, page_content='Q: 가족 여행 가기로 했어\nA: 온 가족이 모두 마음에 드는 곳으로 가보세요.'), Document(id='07cb2e92-a781-4b27-a443-6ab6c76e2e39', metadata={'row': 28, 'label': '0', 'source': 'ChatBotData_sample.csv'}, page_content='Q: 가족끼리 여행간다.\nA: 더 가까워질 기회가 되겠네요.')]


In [13]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

csv_store = Chroma.from_documents(csv_docs, local_emb, persist_directory='chroma_db_csv_local',
                                  collection_configuration={'hnsw': {'space': 'cosine'}})
for d, s in csv_store.similarity_search_with_relevance_scores('가족 여행 갈거야. 기대가 많이 되', k=2):
    print(f"{s:.3f} row {d.metadata['row']:2d} | {d.page_content.splitlines()[0]}")
print("\nfilter {'label': '0'} :", len(csv_store.similarity_search('여행', k=3, filter={'label': '0'})), '개')
print("filter {'label': 0}   :", len(csv_store.similarity_search('여행', k=3, filter={'label': 0})), "개 ← 숫자 0 으로는 못 찾는다")

0.825 row 23 | Q: 가족 여행 가기로 했어
0.797 row 28 | Q: 가족끼리 여행간다.

filter {'label': '0'} : 3 개
filter {'label': 0}   : 0 개 ← 숫자 0 으로는 못 찾는다


- 로컬 임베딩도 수업(OpenAI)과 같은 **row 23 · 28** 을 찾았다
- CSV 는 글자 파일이라 `CSVLoader` 는 모든 값을 **글자**로 넣는다 → 필터는 `{'label': '0'}`. 숫자로 쓰고 싶으면 불러온 뒤 `d.metadata['label'] = int(d.metadata['label'])`
- 챗봇 데이터는 **Q 와 A 를 한 Document 에** 넣었다 — 질문으로 찾고 답을 함께 얻는다(Day 049 의 'QA 한 쌍 = 청크 하나' 와 같은 생각). 분할이 필요 없다

> **📝 고친 부분 (필기 `RAG_실습6` · 교재)**
> - CSV 메타데이터는 글자 — `label` 로 거를 때 `0` 이 아니라 `'0'`
> - 필기도 '폴더가 있으면 불러오기'(`chroma_db_csv`) — 데이터를 바꾸면 폴더 이름도 바꾼다(Day 049 §5)
> - 교재 p.52 연산자 표에 `$in` · `$nin` 이 빠짐(목록 조건에 자주 쓴다) · FAISS 의 '다르다' 는 `$neq`
> - 교재 p.53 `LIMIT 1` ↔ `'k': 1` — `k` 는 **유사도 순서**로 자르는 것
> - 교재 p.54 결론의 근거 : FAISS 필터는 `fetch_k` 개 안에서만 거른다(§5 에서 4개 요청에 1개)

## 정리

| 할 일 | 코드 |
|---|---|
| JSON → Document | `Document(page_content=r['page_content'], metadata=r['metadata'])` |
| CSV → Document | `CSVLoader(path, content_columns=[...], metadata_columns=[...])` — 값은 모두 글자 |
| 조건 + 유사도 | `as_retriever(search_kwargs={'k': 1, 'filter': {'$and': [...]}})` |
| 연산자 | `$eq` `$ne` `$gt` `$gte` `$lt` `$lte` `$in` `$nin` `$and` `$or` (FAISS 는 `$neq`) |